# Phase 2 — Ngày 1–2: Amazon-M2 preprocessing

Đầu ra:

- `product_versions.parquet`: `product_id`, `normalized_text`, `version_index`
- `global_products.parquet`: `product_index`, `product_id`, `version_count`
- `model_sessions_train.parquet`: `prev_items`, `next_item`
- `model_sessions_validation.parquet`: `prev_items`, `next_item`
- `preprocessing_manifest.json`

Quy tắc đã chốt:

- Raw CSV chỉ đọc, không sửa.
- Đọc và xử lý toàn bộ mỗi CSV trong một lần, không chia chunk.
- `locale`, raw `price` và `author` không đi vào model-ready artifacts.
- Giữ mọi metadata row trước bước embedding; không drop duplicate `product_id`.
- Catalog và session item dùng chung namespace global theo `product_id`.

## 0. Cấu hình

In [ ]:
from pathlib import Path

DATASET_ROOT = None
OUTPUT_ROOT = None

VALIDATION_PERCENT = 10
SEED = 2026
COMPUTE_RAW_CHECKSUMS = True
RESET_OUTPUT = True

print("Configuration loaded.")

## 1. Import và tìm Amazon-M2

In [ ]:
import hashlib
import json
import re
import shutil
import unicodedata
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split


def locate_dataset_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if (root / "products_train.csv").is_file() and (root / "sessions_train.csv").is_file():
            return root
        raise FileNotFoundError(f"Missing products_train.csv or sessions_train.csv in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "ai-recommendation" / "dataset" / "original" / "amazon-m2",
        cwd / "dataset" / "original" / "amazon-m2",
        cwd.parent / "dataset" / "original" / "amazon-m2",
    ]
    for candidate in candidates:
        if (candidate / "products_train.csv").is_file() and (candidate / "sessions_train.csv").is_file():
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for products_file in kaggle_input.glob("**/products_train.csv"):
            if (products_file.parent / "sessions_train.csv").is_file():
                return products_file.parent.resolve()
    raise FileNotFoundError("Amazon-M2 was not found. Add the dataset on Kaggle or set DATASET_ROOT.")


DATASET_ROOT = locate_dataset_root(DATASET_ROOT)
if OUTPUT_ROOT is None:
    OUTPUT_ROOT = (
        Path("/kaggle/working/vmarket_phase2_preprocessed")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "vmarket_phase2_preprocessed"
    )
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()

PRODUCTS_CSV = DATASET_ROOT / "products_train.csv"
SESSIONS_CSV = DATASET_ROOT / "sessions_train.csv"

print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("products_train.csv:", f"{PRODUCTS_CSV.stat().st_size / 2**20:,.1f} MiB")
print("sessions_train.csv:", f"{SESSIONS_CSV.stat().st_size / 2**20:,.1f} MiB")

## 2. Kiểm tra schema raw

In [ ]:
RAW_PRODUCT_COLUMNS = {
    "id", "locale", "title", "price", "brand", "color",
    "size", "model", "material", "author", "desc",
}
RAW_SESSION_COLUMNS = {"prev_items", "next_item", "locale"}
PRODUCT_VERSION_COLUMNS = ("product_id", "normalized_text", "version_index")
GLOBAL_PRODUCT_COLUMNS = ("product_index", "product_id", "version_count")
MODEL_SESSION_COLUMNS = ("prev_items", "next_item")
TEXT_FIELDS = ("title", "brand", "color", "size", "material", "model", "desc")
EXCLUDED_MODEL_FIELDS = {"locale", "price", "author"}

products_head = pd.read_csv(PRODUCTS_CSV, nrows=5)
sessions_head = pd.read_csv(SESSIONS_CSV, nrows=5)

missing_product_columns = RAW_PRODUCT_COLUMNS - set(products_head.columns)
missing_session_columns = RAW_SESSION_COLUMNS - set(sessions_head.columns)
assert not missing_product_columns, missing_product_columns
assert not missing_session_columns, missing_session_columns

print("Raw product columns:", products_head.columns.tolist())
print("Raw session columns:", sessions_head.columns.tolist())
display(products_head[["id", "locale", "title"]].head(2))
display(sessions_head.head(2))

## 3. Hàm preprocessing

In [ ]:
QUOTED_TOKEN = re.compile(r"['\"]([^'\"]+)['\"]")
PRODUCT_ID_PATTERN = re.compile(r"^[A-Z0-9]{10}$")
WHITESPACE = re.compile(r"\s+")


def parse_prev_items(value):
    # Parse numpy-array-like text safely; never use eval.
    if isinstance(value, (list, tuple, np.ndarray)):
        tokens = [str(item) for item in value]
    else:
        if value is None or (not isinstance(value, str) and pd.isna(value)):
            raise ValueError("prev_items cannot be null")
        text = str(value).strip()
        if text == "[]":
            return []
        tokens = QUOTED_TOKEN.findall(text)
    if not tokens:
        raise ValueError(f"Could not parse prev_items safely: {str(value)[:120]!r}")
    invalid = [token for token in tokens if not PRODUCT_ID_PATTERN.fullmatch(token)]
    if invalid:
        raise ValueError(f"Invalid product IDs in prev_items: {invalid[:5]}")
    return tokens


def normalize_text(value):
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    return WHITESPACE.sub(" ", value).strip()


def build_product_text(record):
    parts = []
    for field in TEXT_FIELDS:
        value = normalize_text(record.get(field))
        if value:
            label = "description" if field == "desc" else field
            parts.append(f"{label}: {value}")
    return " | ".join(parts)


def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def reset_output_directory(path, enabled=True):
    path = Path(path).resolve()
    if path.exists() and any(path.iterdir()):
        if not enabled:
            raise FileExistsError(f"Output already exists: {path}")
        if path.name != "vmarket_phase2_preprocessed":
            raise ValueError(f"Refusing to delete an output path without the expected safe name: {path}")
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

## 4. Xử lý product metadata

Cell này:

1. Đọc toàn bộ `products_train.csv`.
2. Giữ mọi metadata row.
3. Xóa `locale`, raw `price`, `author` khỏi output bằng cách chỉ tạo đúng ba cột contract.
4. Tạo `normalized_text`.
5. Ghi một file product versions và global product index.

In [ ]:
OUTPUT_ROOT = reset_output_directory(OUTPUT_ROOT, RESET_OUTPUT)

products = pd.read_csv(PRODUCTS_CSV)
missing = RAW_PRODUCT_COLUMNS - set(products.columns)

if missing:
    raise ValueError(f"products_train is missing required columns: {sorted(missing)}")
if products["id"].isna().any() or products["id"].astype("string").str.strip().eq("").any():
    raise ValueError("products_train contains null or empty product_id values")

product_ids = products["id"].astype("string").str.strip().reset_index(drop=True)
product_versions = pd.DataFrame({
    "product_id": product_ids,
    "normalized_text": [build_product_text(record) for record in products.to_dict("records")],
    "version_index": range(len(products)),
})

if product_versions["normalized_text"].eq("").any():
    raise ValueError("At least one product metadata row produced empty normalized_text")
assert tuple(product_versions.columns) == PRODUCT_VERSION_COLUMNS
assert EXCLUDED_MODEL_FIELDS.isdisjoint(product_versions.columns)

product_versions_path = OUTPUT_ROOT / "product_versions.parquet"
product_versions.to_parquet(product_versions_path, index=False, compression="zstd")

version_counts = product_versions["product_id"].value_counts(sort=False)
global_product_ids = sorted(version_counts.index.tolist())
global_products = pd.DataFrame({
    "product_index": range(len(global_product_ids)),
    "product_id": global_product_ids,
    "version_count": version_counts.loc[global_product_ids].to_numpy(),
})
assert tuple(global_products.columns) == GLOBAL_PRODUCT_COLUMNS
global_products.to_parquet(OUTPUT_ROOT / "global_products.parquet", index=False, compression="zstd")

product_rows = len(product_versions)
print(f"product version rows: {product_rows:,}")
print(f"global products: {len(global_products):,}")
print(f"multi-version products: {global_products['version_count'].gt(1).sum():,}")

## 5. Xử lý sessions và kiểm tra global-catalog coverage

Session được tách train/validation ngẫu nhiên bằng `train_test_split`. `random_state=SEED` giúp kết quả có thể tái lập khi chạy lại notebook.

In [ ]:
catalog_ids = set(global_products["product_id"].tolist())
sessions = pd.read_csv(SESSIONS_CSV)

missing = RAW_SESSION_COLUMNS - set(sessions.columns)
if missing:
    raise ValueError(f"sessions_train is missing required columns: {sorted(missing)}")
if sessions["next_item"].isna().any():
    raise ValueError("sessions_train contains null next_item values")

model_sessions = pd.DataFrame({
    "prev_items": sessions["prev_items"].map(parse_prev_items),
    "next_item": sessions["next_item"].astype("string").str.strip(),
})
assert tuple(model_sessions.columns) == MODEL_SESSION_COLUMNS
assert EXCLUDED_MODEL_FIELDS.isdisjoint(model_sessions.columns)
if model_sessions["prev_items"].map(len).eq(0).any():
    raise ValueError("At least one session has an empty history")

missing_count = 0
missing_sample = []
event_count = 0
for prev_items, next_item in model_sessions.itertuples(index=False, name=None):
    event_count += len(prev_items) + 1
    for product_id in (*prev_items, next_item):
        if product_id not in catalog_ids:
            missing_count += 1
            if len(missing_sample) < 20:
                missing_sample.append(product_id)
if missing_count:
    raise ValueError(
        f"{missing_count:,} events are missing from the global catalog; sample={missing_sample}"
    )

train, validation = train_test_split(
    model_sessions,
    test_size=VALIDATION_PERCENT / 100,
    random_state=SEED,
    shuffle=True,
)
train = train.reset_index(drop=True)
validation = validation.reset_index(drop=True)

train_path = OUTPUT_ROOT / "model_sessions_train.parquet"
validation_path = OUTPUT_ROOT / "model_sessions_validation.parquet"
train.to_parquet(train_path, index=False, compression="zstd")
validation.to_parquet(validation_path, index=False, compression="zstd")

train_rows = len(train)
validation_rows = len(validation)
print(f"train sessions: {train_rows:,}")
print(f"validation sessions: {validation_rows:,}")
print(f"events: {event_count:,}")
print("missing catalog events:", missing_count)

## 6. Manifest và contract checks cuối

In [ ]:
raw_files = {
    "products": {"path": str(PRODUCTS_CSV), "size_bytes": PRODUCTS_CSV.stat().st_size},
    "sessions": {"path": str(SESSIONS_CSV), "size_bytes": SESSIONS_CSV.stat().st_size},
}
if COMPUTE_RAW_CHECKSUMS:
    raw_files["products"]["sha256"] = sha256_file(PRODUCTS_CSV)
    raw_files["sessions"]["sha256"] = sha256_file(SESSIONS_CSV)

manifest = {
    "contract_version": "phase2-preprocessing-v2",
    "raw_files": raw_files,
    "configuration": {
        "load_mode": "full",
        "validation_percent": VALIDATION_PERCENT,
        "split_method": "sklearn.model_selection.train_test_split",
        "seed": SEED,
        "excluded_model_fields": sorted(EXCLUDED_MODEL_FIELDS),
    },
    "products": {
        "product_version_rows": product_rows,
        "global_products": len(global_products),
        "multi_version_products": int(global_products["version_count"].gt(1).sum()),
        "columns": list(PRODUCT_VERSION_COLUMNS),
        "versions_file": str(product_versions_path),
        "index_file": str(OUTPUT_ROOT / "global_products.parquet"),
    },
    "sessions": {
        "session_rows": train_rows + validation_rows,
        "train_rows": train_rows,
        "validation_rows": validation_rows,
        "events": event_count,
        "missing_catalog_events": missing_count,
        "columns": list(MODEL_SESSION_COLUMNS),
        "train_file": str(train_path),
        "validation_file": str(validation_path),
    },
}

manifest_path = OUTPUT_ROOT / "preprocessing_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

assert tuple(pq.read_schema(product_versions_path).names) == PRODUCT_VERSION_COLUMNS
assert tuple(pq.read_schema(train_path).names) == MODEL_SESSION_COLUMNS
assert tuple(pq.read_schema(validation_path).names) == MODEL_SESSION_COLUMNS
assert manifest["sessions"]["missing_catalog_events"] == 0
assert product_rows >= len(global_products)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nFinal contract checks: PASSED")

## 7. Xem sample output và bàn giao

In [ ]:
product_sample = pd.read_parquet(product_versions_path).head(5)
global_sample = pd.read_parquet(OUTPUT_ROOT / "global_products.parquet").head(5)
session_sample = pd.read_parquet(train_path).head(5)

display(product_sample)
display(global_sample)
display(session_sample)

output_size = sum(path.stat().st_size for path in OUTPUT_ROOT.rglob("*") if path.is_file())
print(f"Output size: {output_size / 2**20:,.1f} MiB")
print("Output root:", OUTPUT_ROOT)